In [1]:
import pandas as pd

In [ ]:
df_BOM=pd.read_excel(r"Hardware_Share_Folder\BOM_Summarized.xlsx",sheet_name="Sheet1")
df_BOM

In [ ]:
df_BOM['PartNumber']=df_BOM['Finished Good Description'].astype(str).str.split(' ').str[0]
df_BOM = df_BOM[(df_BOM['PartNumber'] != "nan") & (df_BOM['PartNumber'] != "")].reset_index(drop=True)
df_BOM["Qty Per Kit"]=df_BOM["Qty Per Kit"].astype(int).astype(str)
df_BOM = df_BOM[df_BOM['Raw Material Description'] != "."]
df_BOM

In [ ]:
df_Items=pd.read_excel(r"Hardware_Share_Folder\BOM_Summarized.xlsx",sheet_name="Sheet2")
df_Items = df_Items[df_Items['IBI Numbers'].notna()]
df_Items

In [ ]:
df_merged = pd.merge(df_BOM, df_Items['IBI Numbers'], left_on='PartNumber', right_on='IBI Numbers', how='inner')
df_merged

In [12]:
df_merged["PartQty"]=df_merged['Raw Material Description']+" (" + df_merged["Qty Per Kit"]+ ")"
df_merged=df_merged[df_merged['Type']=="PART"]

In [ ]:
df_Cropped=df_merged[['IBI Numbers','PartQty']].drop_duplicates().reset_index(drop=True)
df_final=df_Cropped.groupby('IBI Numbers')['PartQty'].apply(lambda x: '; '.join(x)).reset_index()
df_final

In [ ]:
output_file = r"Hardware_Share_Folder\cleaned_file.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_final.to_excel(writer, sheet_name="Final", index=False)
    df_merged.to_excel(writer, sheet_name="All", index=False)
